In [8]:
import torch
import torch.nn as nn
import numpy as np

CHARACTERS = [
    'Intervention', 'Barrier', 'CrossingSignal',
    'Man', 'Woman', 'Pregnant', 'Stroller', 'OldMan', 'OldWoman',
    'Boy', 'Girl', 'Homeless', 'LargeWoman', 'LargeMan', 'Criminal',
    'MaleExecutive', 'FemaleExecutive', 'FemaleAthlete', 'MaleAthlete',
    'FemaleDoctor', 'MaleDoctor', 'Dog', 'Cat'
]

CHAR_TO_IDX = {char: idx for idx, char in enumerate(CHARACTERS)}

class MoralReasoningTransformer(nn.Module):
    def __init__(
        self,
        num_characters: int = 23,
        max_cardinality: int = 10,
        num_teams: int = 2,
        embed_dim: int = 64,
        num_heads: int = 2,
        num_layers: int = 2,
        dropout: float = 0.1
    ):
        super().__init__()

        self.num_characters = num_characters
        self.embed_dim = embed_dim
        self.char_embed_dim = embed_dim // 2
        self.card_team_embed_dim = embed_dim // 4

        # Compositional embeddings (character gets half, cardinality and team get quarter each)
        self.character_embedding = nn.Embedding(num_characters, self.char_embed_dim)
        self.cardinality_embedding = nn.Embedding(max_cardinality + 1, self.card_team_embed_dim)
        self.team_embedding = nn.Embedding(num_teams, self.card_team_embed_dim)

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # CLS token for aggregation
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))

        # Classification head on CLS token
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, embed_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim // 2, 1)
        )

    def encode_outcome(self, counts, team_id):
        batch_size = counts.shape[0]

        character_ids = torch.arange(
            self.num_characters,
            device=counts.device
        ).unsqueeze(0).expand(batch_size, -1)

        char_emb = self.character_embedding(character_ids)  # (batch, 23, embed_dim//2)
        card_emb = self.cardinality_embedding(counts)        # (batch, 23, embed_dim//4)

        team_id_tensor = torch.full(
            (batch_size, self.num_characters),
            team_id,
            device=counts.device,
            dtype=torch.long
        )
        team_emb = self.team_embedding(team_id_tensor)       # (batch, 23, embed_dim//4)

        # Concatenate along last dimension: embed_dim//2 + embed_dim//4 + embed_dim//4 = embed_dim
        tokens = torch.cat([char_emb, card_emb, team_emb], dim=-1)

        return tokens

    def forward(self, scenarios):
        batch_size = scenarios.shape[0]

        outcome_0 = scenarios[:, 0, :]
        outcome_1 = scenarios[:, 1, :]

        tokens_0 = self.encode_outcome(outcome_0, team_id=0)
        tokens_1 = self.encode_outcome(outcome_1, team_id=1)

        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        all_tokens = torch.cat([cls_tokens, tokens_0, tokens_1], dim=1)

        encoded = self.transformer(all_tokens)
        cls_output = encoded[:, 0, :]
        logits = self.classifier(cls_output)

        return logits


def load_model(path='best_model.pt'):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = MoralReasoningTransformer()
    model.load_state_dict(torch.load(path, map_location=device)['model_state_dict'])
    model.to(device).eval()
    return model, device


def get_probs(model, device, scenario_tuple):
    """
    scenario_tuple: (dict_0, dict_1) where dicts have character names as keys
    Returns: [prob_outcome_0, prob_outcome_1] that sum to 1.0
    """
    outcome_0, outcome_1 = scenario_tuple

    vec_0 = np.zeros(23, dtype=np.int64)
    vec_1 = np.zeros(23, dtype=np.int64)

    for char, count in outcome_0.items():
        vec_0[CHAR_TO_IDX[char]] = count
    for char, count in outcome_1.items():
        vec_1[CHAR_TO_IDX[char]] = count

    scenario_1 = torch.tensor([[vec_0, vec_1]], dtype=torch.long).to(device)
    scenario_2 = torch.tensor([[vec_1, vec_0]], dtype=torch.long).to(device)

    with torch.no_grad():
        probs_1 = torch.sigmoid(model(scenario_1)).item()
        probs_2 = torch.sigmoid(model(scenario_2)).item()
        prob1 = (probs_1 + (1 - probs_2))/2
        prob0 = 1 - prob1
        return [prob0, prob1]


# Usage:
# model, device = load_model('best_model.pt')
# prob = get_probs(model, device, ({'Man': 3}, {'Criminal': 3}))

In [9]:
if __name__ == "__main__":
    # Load model
    model, device = load_model('model_arch_tiny_2layers_2heads.pt')

    # Example 1: Save 3 men and 1 dog vs 2 women and 1 criminal
    scenario1 = (
        {'Man': 3, 'Dog': 1},
        {'Woman': 2, 'Criminal': 1},
    )

    prob1 = get_probs(model, device, scenario1)
    print(f"\nScenario 1:")
    print(f"  Outcome 0: {scenario1[0]}")
    print(f"  Outcome 1: {scenario1[1]}")
    print(f"  P: {prob1}")

    # Example 2: Save 5 men vs 5 criminals
    scenario2 = (
        {'Man': 5},
        {'Criminal': 5}
    )

    prob2 = get_probs(model, device, scenario2)
    print(f"\nScenario 2:")
    print(f"  Outcome 0: {scenario2[0]}")
    print(f"  Outcome 1: {scenario2[1]}")
    print(f"  P: {prob2}")

    # Example 3: Save 1 pregnant woman vs 2 old men
    scenario3 = (
        {'Pregnant': 1},
        {'OldMan': 2}
    )

    prob3 = get_probs(model, device, scenario3)
    print(f"\nScenario 3:")
    print(f"  Outcome 0: {scenario3[0]}")
    print(f"  Outcome 1: {scenario3[1]}")
    print(f"  P: {prob3}")

    # Example 4: Intervention scenario - crossing signal
    scenario4 = (
        {'CrossingSignal': 1, 'Man': 3},
        {'Barrier': 1, 'Woman': 2, 'Boy': 1}
    )

    prob4 = get_probs(model, device, scenario4)
    print(f"\nScenario 4:")
    print(f"  Outcome 0: {scenario4[0]}")
    print(f"  Outcome 1: {scenario4[1]}")
    print(f"  P: {prob4}")

    scenario5 = (
        {"OldMan": 1},
        {"Man": 1}
    )

    prob5 = get_probs(model, device, scenario5)
    print(f"\nScenario 5:")
    print(f"  Outcome 0: {scenario5[0]}")
    print(f"  Outcome 1: {scenario5[1]}")
    print(f"  P: {prob5}")


Scenario 1:
  Outcome 0: {'Man': 3, 'Dog': 1}
  Outcome 1: {'Woman': 2, 'Criminal': 1}
  P: [0.8888750448822975, 0.11112495511770248]

Scenario 2:
  Outcome 0: {'Man': 5}
  Outcome 1: {'Criminal': 5}
  P: [0.7847440391778946, 0.2152559608221054]

Scenario 3:
  Outcome 0: {'Pregnant': 1}
  Outcome 1: {'OldMan': 2}
  P: [0.896774347871542, 0.10322565212845802]

Scenario 4:
  Outcome 0: {'CrossingSignal': 1, 'Man': 3}
  Outcome 1: {'Barrier': 1, 'Woman': 2, 'Boy': 1}
  P: [0.6012613326311111, 0.39873866736888885]

Scenario 5:
  Outcome 0: {'OldMan': 1}
  Outcome 1: {'Man': 1}
  P: [0.18230534344911575, 0.8176946565508842]
